In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import hypergeom, chi2_contingency, fisher_exact
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')

# Read the CSV file
class_frame = pd.read_csv("data/10-2-arms-data-x.csv")
#print("Data summary:")
#print(class_frame.describe(include='all'))
print(class_frame.head())

In [ ]:
print("\nCrosstab of gender and armcross:")
crosstab = pd.crosstab(class_frame['gender'], class_frame['armcross'])
print(crosstab)

In [ ]:
# Calculate observed difference
female_right = crosstab.loc['F', 'R'] if 'R' in crosstab.columns else 0
male_right = crosstab.loc['M', 'R'] if 'R' in crosstab.columns else 0
n_female = crosstab.loc['F'].sum()
n_male = crosstab.loc['M'].sum()
observed_diff = 100 * (female_right/n_female - male_right/n_male)

print(f"\nObserved difference: {observed_diff:.2f}%")

In [ ]:
# Random sampling without replacement - PERMUTATION TEST
# Core idea: "What if we kept everyone's gender the same, but randomly shuffled who crosses their right arm on top?"
# Tests null hypothesis that arm-crossing preference is independent of gender by randomly reassigning 
# arm-crossing behavior while keeping gender fixed.

# Create vector representing all 54 people's arm-crossing behavior
right_arm = np.concatenate([np.zeros(22), np.ones(32)])  # 22 people cross left arm on top (0) + 32 cross right arm on top (1)
diff_sample = np.zeros(1000)  # Store 1000 random proportion differences

np.random.seed(42)  # for reproducibility
for i in range(1000):
    # Randomly shuffle the arm-crossing behaviors (like randomly reassigning arm-crossing labels to people)
    # Gender composition stays the same (14 females, 40 males), but who has which arm-crossing preference is now random
    r = np.random.permutation(right_arm)
    
    # Assign positions 0-13 to females, count how many cross right arm on top
    female_right = np.sum(r[:14])  
    # The remaining right-arm crossers must be male
    male_right = 32 - female_right     
    # Calculate proportion difference for this random shuffle
    diff_sample[i] = 100 * (female_right/14 - male_right/40)

# diff_sample now contains 1000 proportion differences that could occur purely by chance
# if there was no relationship between gender and arm-crossing (empirical null distribution)
plt.figure(figsize=(8, 5))
plt.hist(diff_sample, bins=10, alpha=0.7, edgecolor='black')
plt.xlim(-60, 60)
plt.xlabel('% difference between female and male')
plt.ylabel('Frequency')
plt.title(f'Random permutations')
plt.axvline(x=observed_diff, color='red', linestyle='--', label=f'Observed difference ({observed_diff:.1f}%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calculate p-value (two-tailed)
# Count how many simulated differences are as extreme or more extreme than observed
p_value = np.mean(np.abs(diff_sample) >= np.abs(observed_diff))
print(f"\nTwo-tailed p-value: {p_value:.4f}")

In [ ]:
# Hypergeometric distribution analysis
# Using "balls in urns" analogy to model what would happen if arm-crossing was completely random:
# - Total population: 54 people (14 females + 40 males)
# - "White balls": 14 females 
# - "Black balls": 40 males
# - "Balls drawn": 32 people who cross right arm on top
# - Question: If we randomly select 32 people to cross right arm on top, how many would be female?

x = np.arange(14, -1, -1)  # All possible values for how many females could cross right arm on top [14, 13, 12, ..., 1, 0]
y = hypergeom.pmf(x, 54, 14, 32)  # Probability of each scenario under hypergeometric distribution
                                  # Parameters: total population (54), females (14), right-arm crossers (32)

# Calculate proportion differences for each possible scenario
female_prop = 100 * x / 14      # % of females crossing right arm on top
male_prop = 100 * (32 - x) / 40 # % of males crossing right arm on top  
diff = female_prop - male_prop  # Difference in proportions

# Convert probabilities into counts for a sample of 1000 to create distribution
hyper_count = np.round(1000 * y).astype(int)  # How many times each difference should appear in 1000 samples
cum_hyper_count = np.cumsum(hyper_count)      # Cumulative counts for indexing

# Fill array of 1000 values where each difference appears according to its hypergeometric probability
# This represents what the distribution would look like if arm-crossing was completely random (null hypothesis)
diff_hyper_count = np.zeros(1000)
for i in range(2, len(diff) - 1):  # start at first non-zero element
    if i == 2:
        start_idx = 0
    else:
        start_idx = cum_hyper_count[i-2]
    end_idx = cum_hyper_count[i-1]
    if end_idx > start_idx:
        diff_hyper_count[start_idx:end_idx] = diff[i]


plt.figure(figsize=(8, 5))
plt.hist(diff_hyper_count, bins=11, alpha=0.7, edgecolor='black')
plt.xlim(-60, 60)
plt.xlabel('% difference between female and male')
plt.ylabel('Frequency')
plt.title('All possible permutations')
plt.axvline(x=7, color='red', linestyle='--', label='Observed difference (7%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calculate p-value (two-tailed)
# Sum probabilities of all outcomes as extreme or more extreme than observed
p_value_hyper = np.sum(y[np.abs(diff) >= np.abs(observed_diff)])
print(f"Two-tailed p-value (hypergeometric): {p_value_hyper:.4f}")


# Create histogram and color bars based on p-value region
plt.figure(figsize=(8, 5))
counts, bins, patches = plt.hist(diff_hyper_count, bins=11, edgecolor='black')

# Color each bar based on whether it's in the extreme region
for i, patch in enumerate(patches):
    bin_center = (bins[i] + bins[i+1]) / 2
    if np.abs(bin_center) >= np.abs(observed_diff):
        patch.set_facecolor('coral')  # Extreme values (p-value region)
    else:
        patch.set_facecolor('steelblue')  # Non-extreme values

plt.xlim(-60, 60)
plt.xlabel('% difference between female and male')
plt.ylabel('Frequency')
plt.title(f'All possible permutations (p-value = {p_value_hyper:.4f})')
plt.axvline(x=observed_diff, color='red', linestyle='--', linewidth=2, 
            label=f'Observed difference ({observed_diff:.1f}%)')
plt.axvline(x=-observed_diff, color='red', linestyle='--', linewidth=2, alpha=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', edgecolor='black', label='Non-extreme differences'),
    Patch(facecolor='coral', edgecolor='black', label='Extreme differences (p-value region)'),
    plt.Line2D([0], [0], color='red', linestyle='--', linewidth=2, label=f'Observed difference ({observed_diff:.1f}%)')
]
plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*50)
print("HYPOTHESIS TESTS")
print("="*50)

# Create 2×2 contingency table showing observed data:
# |           | Left arm on top | Right arm on top | Total |
# |-----------|-----------------|------------------|-------|
# | Female    | 5               | 9                | 14    |
# | Male      | 17              | 23               | 40    |
# | Total     | 22              | 32               | 54    |
x = np.array([[5, 9], [17, 23]])   # Females: [5 left, 9 right], Males: [17 left, 23 right]

# 1. CHI-SQUARE TEST
# Tests whether gender and arm-crossing preference are independent (no association)
# Compares observed counts to expected counts if there was no association
# Large differences → large chi-square statistic → small p-value
# Null hypothesis: Gender and arm-crossing are independent

# WHAT IS THE CHI-SQUARE STATISTIC?
# The chi-square statistic is a single number that quantifies how much the observed data 
# differs from what we'd expect if there was no association between the two variables.
# 
# Mathematical Formula: χ² = Σ [(Observed - Expected)² / Expected]
# For each cell: (1) Calculate difference, (2) Square it, (3) Divide by expected, (4) Sum all cells
#
# Small χ² → observed data close to expected → little evidence against null hypothesis
# Large χ² → observed data very different from expected → strong evidence for association

chi2_stat, chi2_p, chi2_dof, chi2_expected = chi2_contingency(x)
print(f"\nChi-square test:")
print(f"Chi-square statistic: {chi2_stat:.4f}")
#print(f"p-value: {chi2_p:.4f}")
#print(f"Degrees of freedom: {chi2_dof}")
print(f"Expected frequencies:")
print(f"{chi2_expected}")

# EXPLICIT CHI-SQUARE CALCULATION (to show how it works)
print(f"\nManual chi-square calculation:")
print(f"Observed table:")
print(f"{x}")
print(f"Expected table:")
print(f"{chi2_expected}")

# Calculate chi-square manually for each cell
chi2_manual = 0
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        observed = x[i, j]
        expected = chi2_expected[i, j]  # Use the actual expected value, not rounded
        cell_contribution = (observed - expected)**2 / expected
        print(f"Cell ({i},{j}): (({observed} - {expected:.6f})² / {expected:.6f}) = {cell_contribution:.6f}")
        chi2_manual += cell_contribution

print(f"Total χ² = {chi2_manual:.6f}")
print(f"Matches scipy result: {abs(chi2_manual - chi2_stat) < 1e-10}")

# BE CAREFUL WITH DEFAULT VALUES

Here is the [documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html) for the Chi-square test of independence of variables in a contingency table.

And here is the [explanation from Claude](https://claude.ai/share/4797b6e2-2a9d-4718-a37e-44313cb58135)

In [ ]:
# Manual calculation WITH Yates' correction
chi2_manual_yates = 0
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        observed = x[i, j]
        expected = chi2_expected[i, j]
        # Apply Yates' correction: subtract 0.5 from absolute difference
        cell_contribution = (abs(observed - expected) - 0.5)**2 / expected
        chi2_manual_yates += cell_contribution

print(f"Total χ² = {chi2_manual_yates:.6f}")
print(f"Matches scipy result: {abs(chi2_manual_yates - chi2_stat) < 1e-10}")

In [ ]:
chi2_stat, chi2_p, chi2_dof, chi2_expected = chi2_contingency(x, correction=False)

print(f"Chi-square statistic (without Yates'correction): {chi2_stat:.4f}")
print(f"Total χ² = {chi2_manual:.6f}")
print(f"Matches scipy result: {abs(chi2_manual - chi2_stat) < 1e-10}")

In [ ]:
# 2. TWO-PROPORTION Z-TEST  
# Tests whether the proportion of females crossing right arm on top differs significantly from males
# Female proportion: 9/14 = 64.3%, Male proportion: 23/40 = 57.5%, Difference: 6.8%
# Null hypothesis: The two proportions are equal
successes = np.array([9, 23])  # right arm on top
totals = np.array([14, 40])    # total in each group
z_stat, prop_p = proportions_ztest(successes, totals)
print(f"\nTwo-proportion z-test:")
print(f"Z statistic: {z_stat:.4f}")
print(f"p-value: {prop_p:.4f}")

# 3. FISHER'S EXACT TEST
# Same question as chi-square, but uses exact probabilities instead of approximations
# More accurate for small sample sizes; calculates exact probability under null hypothesis
# Odds ratio: How much more likely females are to cross right arm on top compared to males
odds_ratio, fisher_p = fisher_exact(x)
print(f"\nFisher's exact test:")
print(f"Odds ratio: {odds_ratio:.4f}")
print(f"p-value: {fisher_p:.4f}")

# 4. HYPERGEOMETRIC PROBABILITY CALCULATION
# Calculates probability of observing 9 or more females crossing right arm on top if arm-crossing was random
# Uses same hypergeometric setup as earlier visualization
# Parameters: 8 (we want P(X≥9) = 1-P(X≤8)), 54 (total people), 14 (females), 32 (right-arm crossers)
hyper_p = 1 - hypergeom.cdf(8, 54, 14, 32)
print(f"\nHypergeometric test:")
print(f"P(difference >= 7%) = P(female right >= 9): {hyper_p:.4f}")

# INTERPRETATION:
# Small p-values (< 0.05) suggest the difference is statistically significant
# Large p-values (≥ 0.05) suggest the difference could easily be due to chance

Done with Claude